# Process PI Historian Data

Extracts `PIData.zip` from the lakehouse `Files/PIData/` folder, decompresses hundreds of `.json.gz` files,
flattens the PI tag time-series data, and writes to a Delta table `pi_data`.

**Schema per JSON file:**
- `Tag`, `WebId`, `DataServer`, `WindowStart`, `WindowEnd`, `Count`
- `Items[]`: `{Timestamp, Value, Good, Questionable}`
  - Value can be numeric (float) or status object `{Name, Value, IsSystem}`

In [ ]:
import zipfile
import gzip
import json
import os
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, BooleanType, TimestampType
from pyspark.sql import functions as F

ZIP_PATH = "/lakehouse/default/Files/PIData/PIData.zip"
EXTRACT_DIR = "/tmp/pidata_extract"

# Verify the zip exists
size_mb = os.path.getsize(ZIP_PATH) / (1024 * 1024)
print(f"PIData.zip: {size_mb:.1f} MB")

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

def process_file(name, raw_data):
    """Decompress .json.gz and parse into row tuples."""
    json_bytes = gzip.decompress(raw_data)
    doc = json.loads(json_bytes)

    tag = doc.get("Tag", "")
    web_id = doc.get("WebId", "")
    data_server = doc.get("DataServer", "")

    items = doc.get("Items", [])
    if not items:
        return ([], True, None)

    file_rows = []
    for item in items:
        ts = item.get("Timestamp", "")
        val = item.get("Value")
        good = item.get("Good", None)
        questionable = item.get("Questionable", None)

        if isinstance(val, dict):
            value_numeric = float(val.get("Value")) if val.get("Value") is not None else None
            value_string = val.get("Name", "")
            is_system = val.get("IsSystem", None)
        elif isinstance(val, (int, float)):
            value_numeric = float(val)
            value_string = None
            is_system = None
        elif isinstance(val, str):
            value_numeric = None
            value_string = val
            is_system = None
        else:
            value_numeric = None
            value_string = str(val) if val is not None else None
            is_system = None

        file_rows.append((
            tag, web_id, data_server, ts,
            value_numeric, value_string, is_system,
            good, questionable
        ))

    return (file_rows, False, None)

# Read all compressed bytes from zip (sequential — ZipFile isn't thread-safe)
with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
    gz_files = [n for n in zf.namelist() if n.endswith('.json.gz')]
    total = len(gz_files)
    print(f"Found {total} .json.gz files in the zip")
    file_data = [(name, zf.read(name)) for name in gz_files]

# Parallel decompress + parse
rows = []
file_count = 0
empty_count = 0
error_count = 0

with ThreadPoolExecutor(max_workers=8) as executor:
    futures = {executor.submit(process_file, name, data): name for name, data in file_data}
    done = 0
    for future in as_completed(futures):
        done += 1
        try:
            file_rows, is_empty, _ = future.result()
            if is_empty:
                empty_count += 1
            else:
                rows.extend(file_rows)
                file_count += 1
        except Exception as e:
            error_count += 1
            if error_count <= 5:
                print(f"  Error processing {futures[future]}: {e}")

        if done % 200 == 0:
            print(f"  Processed {done}/{total} files, {len(rows):,} rows...")

print(f"\nDone: {file_count} files with data, {empty_count} empty, {error_count} errors")
print(f"Total rows: {len(rows):,}")

In [ ]:
import pyarrow as pa
import pyarrow.parquet as pq
import shutil

# Write parquet chunks directly to the lakehouse Files mount so Spark can read them.
# We skip the /tmp staging + shutil.copytree bridge: copytree was failing on the
# FUSE mount because the grandparent dir was not guaranteed to exist.
LAKEHOUSE_TEMP = "/lakehouse/default/Files/tmp/pidata_parquet"
os.makedirs("/lakehouse/default/Files/tmp", exist_ok=True)
if os.path.exists(LAKEHOUSE_TEMP):
    shutil.rmtree(LAKEHOUSE_TEMP)
os.makedirs(LAKEHOUSE_TEMP)

arrow_schema = pa.schema([
    ("Tag", pa.string()),
    ("WebId", pa.string()),
    ("DataServer", pa.string()),
    ("Timestamp", pa.string()),
    ("ValueNumeric", pa.float64()),
    ("ValueString", pa.string()),
    ("IsSystem", pa.bool_()),
    ("Good", pa.bool_()),
    ("Questionable", pa.bool_()),
])

CHUNK = 500_000
total_chunks = (len(rows) + CHUNK - 1) // CHUNK
for i in range(0, len(rows), CHUNK):
    chunk = rows[i:i+CHUNK]
    cols = list(zip(*chunk))
    table = pa.table({
        "Tag": cols[0], "WebId": cols[1], "DataServer": cols[2],
        "Timestamp": cols[3], "ValueNumeric": cols[4], "ValueString": cols[5],
        "IsSystem": cols[6], "Good": cols[7], "Questionable": cols[8],
    }, schema=arrow_schema)
    pq.write_table(table, f"{LAKEHOUSE_TEMP}/part_{i//CHUNK:04d}.parquet")
    idx = i // CHUNK + 1
    if idx == 1 or idx == total_chunks or idx % 20 == 0:
        print(f"  Wrote chunk {idx}/{total_chunks} ({len(chunk):,} rows)", flush=True)

# Free the in-memory list before we let Spark scan parquet
del rows
import gc; gc.collect()

# Read back with Spark from lakehouse path. Strip trailing Z and let Spark's default
# ISO parser handle both RV2 (integer seconds) and RV3 (7-digit fractional ticks).
df = spark.read.parquet("Files/tmp/pidata_parquet")
df = df.withColumn("Timestamp", F.to_timestamp(F.regexp_replace("Timestamp", "Z$", "")))

print(f"DataFrame: {df.count():,} rows, {len(df.columns)} columns", flush=True)
df.printSchema()
df.show(10, truncate=False)


In [ ]:
# Write to Delta table
TABLE_NAME = "pi_data"

df.write.mode("overwrite").format("delta").saveAsTable(TABLE_NAME)

print(f"Wrote {TABLE_NAME} to lakehouse")

In [ ]:
# Verify and summary stats
result = spark.sql(f"SELECT COUNT(*) as total_rows, COUNT(DISTINCT Tag) as unique_tags FROM {TABLE_NAME}")
result.show()

# Data quality breakdown
spark.sql(f"""
    SELECT 
        COUNT(*) as total,
        SUM(CASE WHEN Good = true THEN 1 ELSE 0 END) as good_count,
        SUM(CASE WHEN Good = false THEN 1 ELSE 0 END) as bad_count,
        SUM(CASE WHEN Questionable = true THEN 1 ELSE 0 END) as questionable_count,
        SUM(CASE WHEN ValueString IS NOT NULL THEN 1 ELSE 0 END) as status_values,
        SUM(CASE WHEN ValueNumeric IS NOT NULL THEN 1 ELSE 0 END) as numeric_values,
        MIN(Timestamp) as earliest,
        MAX(Timestamp) as latest
    FROM {TABLE_NAME}
""").show(truncate=False)

# Top 10 tags by row count
spark.sql(f"""
    SELECT Tag, COUNT(*) as row_count
    FROM {TABLE_NAME}
    GROUP BY Tag
    ORDER BY row_count DESC
    LIMIT 10
""").show(truncate=False)

## Tags Metadata
Load PI tag metadata CSV and save as a Delta table.

In [ ]:
df_tags = (spark.read.option("header", True).option("inferSchema", True)
    .option("multiLine", True).option("escape", '"')
    .csv("Files/PIData/tags-metadata.csv")
)

df_tags.write.mode("overwrite").format("delta").saveAsTable("pi_tags_metadata")
print(f"pi_tags_metadata: {df_tags.count()} rows, {len(df_tags.columns)} columns")
df_tags.printSchema()
df_tags.show(10, truncate=40)